# 📊 Online Retail – Step 3: Exploratory Data Analysis (EDA)

**Dataset:** Online Retail 2009–2010 (cleaned & outlier-treated from Step 2)  
**Goal:** Khám phá patterns, trends, và insights từ cleaned data.

| Cell | Phân tích |
|------|----------|
| 1 | Setup & Load Cleaned Data |
| 2 | Univariate Analysis – Numerical |
| 3 | Univariate Analysis – Categorical |
| 4 | Bivariate Analysis – Correlation & Relationships |
| 5 | Multivariate Analysis – Patterns & Segmentation |
| 6 | Business Insights & Recommendations |

> ✅ **Yêu cầu:** `df_clean` đã được tạo từ Step 2, hoặc upload lại file `online_retail_cleaned.csv`.

---
## ⚙️ CELL 1: Setup & Load Cleaned Data

**Mục đích:** Import thư viện, load cleaned dataset từ Step 2, cấu hình style cho toàn bộ notebook.

In [ ]:
# ============================================================
# CELL 1: SETUP & LOAD CLEANED DATA
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# ── Global plot style ─────────────────────────────────────────
PALETTE   = ['#1A73E8','#34A853','#FBBC04','#EA4335','#9C27B0',
             '#00BCD4','#FF5722','#607D8B','#E91E63','#4CAF50']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F8F9FA',
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
})

# ── Load data ─────────────────────────────────────────────────
try:
    _ = df_clean
    print("✅ df_clean found in session from Step 2.")
except NameError:
    print("⚠️  df_clean not found. Uploading cleaned CSV...")
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df_clean = pd.read_csv(fname, parse_dates=['InvoiceDate'])
    print(f"✅ Loaded: {fname}")

# ── Ensure key columns exist ──────────────────────────────────
if 'Revenue' not in df_clean.columns:
    df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']
if 'Year' not in df_clean.columns:
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')
    df_clean['Year']    = df_clean['InvoiceDate'].dt.year
    df_clean['Month']   = df_clean['InvoiceDate'].dt.month
    df_clean['Day']     = df_clean['InvoiceDate'].dt.day
    df_clean['Weekday'] = df_clean['InvoiceDate'].dt.day_name()
    df_clean['Hour']    = df_clean['InvoiceDate'].dt.hour

# ── Quick snapshot ────────────────────────────────────────────
print(f"\n📐 Shape          : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"📅 Date range     : {df_clean['InvoiceDate'].min().date()} → {df_clean['InvoiceDate'].max().date()}")
print(f"🌍 Countries      : {df_clean['Country'].nunique()}")
print(f"👤 Customers      : {df_clean['CustomerID'].nunique():,}")
print(f"📦 Products       : {df_clean['StockCode'].nunique():,}")
print(f"🧾 Invoices       : {df_clean['InvoiceNo'].nunique():,}")
print(f"💰 Total Revenue  : £{df_clean['Revenue'].sum():,.2f}")

# ── Analytical subsets ────────────────────────────────────────
num_cols = ['Quantity','UnitPrice','Revenue']
cat_cols = ['Country','Weekday','Year','Month']

display(df_clean.head(3))
print("\n✅ Cell 1 complete – ready for EDA!")

---
## 📊 CELL 2: Univariate Analysis – Numerical Columns

**Mục đích:** Phân tích phân phối của từng cột số (Quantity, UnitPrice, Revenue) — histogram, KDE, boxplot, QQ-plot, và summary statistics đầy đủ.

In [ ]:
# ============================================================
# CELL 2: UNIVARIATE ANALYSIS – NUMERICAL
# ============================================================
from scipy.stats import skew, kurtosis, shapiro

print("=" * 65)
print("📊  UNIVARIATE ANALYSIS – NUMERICAL COLUMNS")
print("=" * 65)

# ── 2A. Extended summary statistics ───────────────────────────
print("\n📋 Extended Summary Statistics:")
stats_rows = []
for col in num_cols:
    s = df_clean[col].dropna()
    stats_rows.append({
        'Column'   : col,
        'Mean'     : round(s.mean(),   2),
        'Median'   : round(s.median(), 2),
        'Std'      : round(s.std(),    2),
        'Min'      : round(s.min(),    2),
        'Max'      : round(s.max(),    2),
        'Skewness' : round(skew(s),    3),
        'Kurtosis' : round(kurtosis(s),3),
        'p25'      : round(s.quantile(0.25), 2),
        'p75'      : round(s.quantile(0.75), 2),
        'p99'      : round(s.quantile(0.99), 2),
    })
display(pd.DataFrame(stats_rows).set_index('Column'))

# ── 2B. Distribution plots (histogram + KDE + boxplot) ────────
fig, axes = plt.subplots(len(num_cols), 3, figsize=(18, 5*len(num_cols)))
fig.suptitle('Numerical Distributions – Cleaned Data', fontsize=16, fontweight='bold', y=1.01)

colors = ['#1A73E8', '#34A853', '#EA4335']
for i, col in enumerate(num_cols):
    s   = df_clean[col].dropna()
    p99 = s.quantile(0.99)
    s_v = s[s <= p99]   # trim for visualization only
    c   = colors[i]

    # Histogram + KDE
    axes[i,0].hist(s_v, bins=60, color=c, alpha=0.75, edgecolor='white', density=True)
    s_v.plot.kde(ax=axes[i,0], color='black', lw=1.8)
    axes[i,0].axvline(s.mean(),   color='red',    linestyle='--', lw=1.5, label=f'Mean {s.mean():.1f}')
    axes[i,0].axvline(s.median(), color='orange', linestyle='-',  lw=1.5, label=f'Median {s.median():.1f}')
    axes[i,0].set_title(f'{col} – Histogram & KDE (≤p99)')
    axes[i,0].set_xlabel(col); axes[i,0].set_ylabel('Density')
    axes[i,0].legend()

    # Boxplot
    bp = axes[i,1].boxplot(s, patch_artist=True, notch=True,
                            boxprops=dict(facecolor=c, alpha=0.6),
                            medianprops=dict(color='black', linewidth=2),
                            flierprops=dict(marker='o', markersize=2, alpha=0.3, color=c))
    axes[i,1].set_title(f'{col} – Boxplot (full range)')
    axes[i,1].set_ylabel(col)

    # QQ-Plot
    stats.probplot(s_v, dist='norm', plot=axes[i,2])
    axes[i,2].set_title(f'{col} – Q-Q Plot (normality check)')
    axes[i,2].get_lines()[0].set(color=c, markersize=3, alpha=0.4)
    axes[i,2].get_lines()[1].set(color='red', linewidth=1.5)

plt.tight_layout()
plt.savefig('univariate_numerical.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 2C. Normality interpretation ──────────────────────────────
print("\n📐 Skewness Interpretation:")
for col in num_cols:
    sk = skew(df_clean[col].dropna())
    label = 'Highly right-skewed (long tail)' if sk > 1 else \
            'Moderately right-skewed'          if sk > 0.5 else \
            'Approximately symmetric'          if abs(sk) < 0.5 else 'Left-skewed'
    print(f"   {col:15s} skewness={sk:.3f}  → {label}")

print("\n✅ Cell 2 complete.")

---
## 🏷️ CELL 3: Univariate Analysis – Categorical Columns

**Mục đích:** Phân tích tần suất các biến phân loại — Country, Weekday, Month, Year — với bar charts, pie charts và phân tích concentration.

In [ ]:
# ============================================================
# CELL 3: UNIVARIATE ANALYSIS – CATEGORICAL
# ============================================================

print("=" * 65)
print("🏷️   UNIVARIATE ANALYSIS – CATEGORICAL COLUMNS")
print("=" * 65)

# ── 3A. Country analysis ──────────────────────────────────────
country_rev  = df_clean.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
country_ord  = df_clean.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)
top10_c      = country_rev.head(10)
top10_orders = country_ord.head(10)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Country Analysis', fontsize=15, fontweight='bold')

# Revenue by country (bar)
bars = axes[0].barh(top10_c.index[::-1], top10_c.values[::-1]/1e6, color=PALETTE[:10])
axes[0].set_title('Top 10 Countries – Total Revenue')
axes[0].set_xlabel('Revenue (£M)')
for bar, val in zip(bars, top10_c.values[::-1]):
    axes[0].text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                 f'£{val/1e6:.2f}M', va='center', fontsize=8)

# Orders by country (bar)
axes[1].barh(top10_orders.index[::-1], top10_orders.values[::-1], color=PALETTE[:10])
axes[1].set_title('Top 10 Countries – Number of Invoices')
axes[1].set_xlabel('Number of Invoices')

# Revenue share pie
other_rev = country_rev[10:].sum()
pie_data  = pd.concat([top10_c, pd.Series({'Others': other_rev})])
axes[2].pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
            colors=PALETTE[:10]+['#BDBDBD'], startangle=90,
            textprops={'fontsize':8})
axes[2].set_title('Revenue Share by Country')

plt.tight_layout()
plt.savefig('univariate_country.png', dpi=120, bbox_inches='tight')
plt.show()

uk_pct = top10_c.get('United Kingdom', country_rev.get('United Kingdom', 0)) / country_rev.sum() * 100
print(f"   🇬🇧 United Kingdom share of total revenue : {uk_pct:.1f}%")
print(f"   🌍 Number of international countries       : {df_clean['Country'].nunique() - 1}")

# ── 3B. Time-based frequency ──────────────────────────────────
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_names   = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                 7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

fig2, axes2 = plt.subplots(2, 2, figsize=(18, 10))
fig2.suptitle('Temporal Frequency Analysis', fontsize=15, fontweight='bold')

# Invoices by weekday
wd_counts = df_clean['Weekday'].value_counts().reindex(weekday_order).fillna(0)
bars_wd   = axes2[0,0].bar(wd_counts.index, wd_counts.values, color=PALETTE[:7])
axes2[0,0].set_title('Transactions by Day of Week')
axes2[0,0].set_ylabel('Number of Transactions')
axes2[0,0].tick_params(axis='x', rotation=30)
for bar in bars_wd:
    axes2[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                    f'{int(bar.get_height()):,}', ha='center', fontsize=8)

# Revenue by weekday
wd_rev = df_clean.groupby('Weekday')['Revenue'].sum().reindex(weekday_order).fillna(0)
axes2[0,1].bar(wd_rev.index, wd_rev.values/1e3, color=PALETTE[:7])
axes2[0,1].set_title('Revenue by Day of Week')
axes2[0,1].set_ylabel('Revenue (£K)')
axes2[0,1].tick_params(axis='x', rotation=30)

# Revenue by month
mo_rev = df_clean.groupby('Month')['Revenue'].sum()
mo_rev.index = [month_names[m] for m in mo_rev.index]
axes2[1,0].bar(mo_rev.index, mo_rev.values/1e3, color=PALETTE[:12])
axes2[1,0].set_title('Revenue by Month')
axes2[1,0].set_ylabel('Revenue (£K)')
axes2[1,0].tick_params(axis='x', rotation=30)

# Invoices by hour
hr_counts = df_clean['Hour'].value_counts().sort_index()
axes2[1,1].bar(hr_counts.index, hr_counts.values, color='#1A73E8', edgecolor='white', alpha=0.85)
axes2[1,1].set_title('Transactions by Hour of Day')
axes2[1,1].set_xlabel('Hour'); axes2[1,1].set_ylabel('Number of Transactions')
axes2[1,1].set_xticks(range(0,24))

plt.tight_layout()
plt.savefig('univariate_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 3C. Top products ──────────────────────────────────────────
top_products_rev = df_clean.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(15)
top_products_qty = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(15)

fig3, axes3 = plt.subplots(1, 2, figsize=(20, 7))
fig3.suptitle('Top 15 Products', fontsize=15, fontweight='bold')

axes3[0].barh(top_products_rev.index[::-1], top_products_rev.values[::-1]/1e3, color='#1A73E8')
axes3[0].set_title('By Revenue (£K)'); axes3[0].set_xlabel('Revenue (£K)')
axes3[0].tick_params(axis='y', labelsize=8)

axes3[1].barh(top_products_qty.index[::-1], top_products_qty.values[::-1]/1e3, color='#34A853')
axes3[1].set_title('By Quantity Sold (K units)'); axes3[1].set_xlabel('Quantity (K)')
axes3[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('univariate_products.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n✅ Cell 3 complete.")

---
## 🔗 CELL 4: Bivariate Analysis – Correlations & Relationships

**Mục đích:** Phân tích tương quan giữa các biến số (Pearson & Spearman), scatter plots, revenue vs quantity theo country, và cross-variable comparisons.

In [ ]:
# ============================================================
# CELL 4: BIVARIATE ANALYSIS
# ============================================================

print("=" * 65)
print("🔗  BIVARIATE ANALYSIS – CORRELATION & RELATIONSHIPS")
print("=" * 65)

# ── 4A. Correlation matrix ─────────────────────────────────────
num_data = df_clean[['Quantity','UnitPrice','Revenue']].dropna()
pearson_corr  = num_data.corr(method='pearson')
spearman_corr = num_data.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Correlation Matrices – Cleaned Data', fontsize=14, fontweight='bold')

mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
sns.heatmap(pearson_corr,  annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, square=True, ax=axes[0],
            cbar_kws={'shrink':0.8}, linewidths=1)
axes[0].set_title('Pearson Correlation')

sns.heatmap(spearman_corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, square=True, ax=axes[1],
            cbar_kws={'shrink':0.8}, linewidths=1)
axes[1].set_title('Spearman Correlation')

plt.tight_layout()
plt.savefig('bivariate_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 4B. Correlation significance ──────────────────────────────
print("\n📐 Pearson Correlation with p-values:")
col_pairs = [('Quantity','UnitPrice'),('Quantity','Revenue'),('UnitPrice','Revenue')]
for c1, c2 in col_pairs:
    r, p = pearsonr(num_data[c1], num_data[c2])
    sig  = '*** p<0.001' if p<0.001 else '** p<0.01' if p<0.01 else '* p<0.05' if p<0.05 else 'not significant'
    print(f"   {c1:12s} × {c2:12s}  r={r:.4f}  {sig}")

# ── 4C. Scatter matrix ─────────────────────────────────────────
sample = df_clean[['Quantity','UnitPrice','Revenue']].sample(min(4000,len(df_clean)), random_state=42)
fig2 = px.scatter_matrix(
    sample,
    dimensions=['Quantity','UnitPrice','Revenue'],
    title='Scatter Matrix – Cleaned Numerical Variables',
    opacity=0.4,
    color_discrete_sequence=['#1A73E8']
)
fig2.update_traces(marker=dict(size=3))
fig2.update_layout(height=550)
fig2.show()

# ── 4D. Revenue by Country – top 10 ───────────────────────────
top10_countries = df_clean.groupby('Country')['Revenue'].sum().nlargest(10).index
df_top10        = df_clean[df_clean['Country'].isin(top10_countries)]

fig3, axes3 = plt.subplots(1, 2, figsize=(18, 6))
fig3.suptitle('Revenue vs Quantity – by Country (Top 10)', fontsize=14, fontweight='bold')

country_agg = df_top10.groupby('Country').agg(
    Revenue=('Revenue','sum'), Quantity=('Quantity','sum'), Orders=('InvoiceNo','nunique')
).reset_index()

axes3[0].scatter(country_agg['Quantity']/1e3, country_agg['Revenue']/1e3,
                 s=country_agg['Orders']/5, alpha=0.7, c=PALETTE[:10], edgecolors='white', lw=0.5)
for _, row in country_agg.iterrows():
    axes3[0].annotate(row['Country'], (row['Quantity']/1e3, row['Revenue']/1e3),
                      textcoords='offset points', xytext=(4,4), fontsize=7)
axes3[0].set_xlabel('Total Quantity (K units)'); axes3[0].set_ylabel('Total Revenue (£K)')
axes3[0].set_title('Country: Quantity vs Revenue (bubble=orders)')

avg_order_val = df_top10.groupby('Country')['Revenue'].mean().sort_values(ascending=False)
axes3[1].barh(avg_order_val.index[::-1], avg_order_val.values[::-1], color=PALETTE[:10])
axes3[1].set_xlabel('Avg Revenue per Transaction (£)')
axes3[1].set_title('Avg Transaction Value by Country')

plt.tight_layout()
plt.savefig('bivariate_country.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 4E. Revenue vs Month (line + bar) ─────────────────────────
monthly = df_clean.groupby(['Year','Month']).agg(
    Revenue=('Revenue','sum'),
    Orders =('InvoiceNo','nunique'),
    AvgOrder=('Revenue','mean')
).reset_index()
monthly['Period'] = monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str).str.zfill(2)

fig4 = make_subplots(specs=[[{'secondary_y': True}]])
fig4.add_trace(go.Bar(x=monthly['Period'], y=monthly['Revenue']/1e3,
                      name='Revenue (£K)', marker_color='#1A73E8', opacity=0.7), secondary_y=False)
fig4.add_trace(go.Scatter(x=monthly['Period'], y=monthly['Orders'],
                           mode='lines+markers', name='# Invoices',
                           line=dict(color='#EA4335', width=2),
                           marker=dict(size=6)), secondary_y=True)
fig4.update_layout(title='Monthly Revenue & Order Count', height=450,
                   xaxis_tickangle=-45, legend=dict(x=0.01, y=0.99))
fig4.update_yaxes(title_text='Revenue (£K)', secondary_y=False)
fig4.update_yaxes(title_text='# Invoices',   secondary_y=True)
fig4.show()

print("\n✅ Cell 4 complete.")

---
## 🔮 CELL 5: Multivariate Analysis – Patterns & Segmentation

**Mục đích:** Khám phá patterns phức tạp qua nhiều biến — heatmaps theo thời gian, RFM segmentation, cohort analysis, top-product phân tích, và interactive treemap.

In [ ]:
# ============================================================
# CELL 5: MULTIVARIATE ANALYSIS – PATTERNS & SEGMENTATION
# ============================================================

print("=" * 65)
print("🔮  MULTIVARIATE ANALYSIS")
print("=" * 65)

# ── 5A. Revenue Heatmap: Weekday × Month ──────────────────────
pivot_wm = df_clean.pivot_table(
    values='Revenue', index='Weekday', columns='Month',
    aggfunc='sum'
).reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
pivot_wm.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig_hm, ax_hm = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_wm/1e3, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax_hm, cbar_kws={'label':'Revenue (£K)'})
ax_hm.set_title('Revenue Heatmap: Weekday × Month (£K)', fontsize=14, fontweight='bold')
ax_hm.set_xlabel('Month'); ax_hm.set_ylabel('Weekday')
plt.tight_layout()
plt.savefig('multivariate_heatmap_weekday_month.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 5B. Hourly revenue heatmap: Hour × Weekday ────────────────
pivot_hw = df_clean.pivot_table(
    values='Revenue', index='Hour', columns='Weekday', aggfunc='sum'
).reindex(columns=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])

fig_hw, ax_hw = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot_hw/1e3, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.3, ax=ax_hw, cbar_kws={'label':'Revenue (£K)'})
ax_hw.set_title('Revenue Heatmap: Hour × Weekday (£K)', fontsize=14, fontweight='bold')
ax_hw.set_xlabel('Weekday'); ax_hw.set_ylabel('Hour of Day')
plt.tight_layout()
plt.savefig('multivariate_heatmap_hour_weekday.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 5C. RFM Segmentation ──────────────────────────────────────
print("\n📊 RFM Segmentation (Recency-Frequency-Monetary):")
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_clean[df_clean['CustomerID'] != 0].groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()

# Score 1–4 (quartile-based)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

def rfm_segment(score):
    if score >= 10: return '🌟 Champions'
    elif score >= 8: return '💛 Loyal'
    elif score >= 6: return '🔵 Potential'
    elif score >= 4: return '🟡 At-Risk'
    else:            return '🔴 Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)
seg_summary    = rfm.groupby('Segment').agg(
    Customers=('CustomerID','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean')
).round(1)
display(seg_summary)

fig_rfm, axes_rfm = plt.subplots(1, 3, figsize=(18, 6))
fig_rfm.suptitle('RFM Segmentation Analysis', fontsize=14, fontweight='bold')

seg_counts = rfm['Segment'].value_counts()
axes_rfm[0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
                colors=PALETTE[:len(seg_counts)], startangle=90, textprops={'fontsize':9})
axes_rfm[0].set_title('Customer Segment Distribution')

seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
axes_rfm[1].bar(seg_rev.index, seg_rev.values/1e3, color=PALETTE[:len(seg_rev)])
axes_rfm[1].set_title('Total Revenue by Segment (£K)')
axes_rfm[1].set_ylabel('Revenue (£K)')
axes_rfm[1].tick_params(axis='x', rotation=30)

axes_rfm[2].scatter(rfm['Frequency'], rfm['Monetary']/1e3, c=rfm['R_Score'],
                    cmap='RdYlGn', alpha=0.5, s=20)
axes_rfm[2].set_title('Frequency vs Monetary (color=Recency score)')
axes_rfm[2].set_xlabel('Frequency (orders)'); axes_rfm[2].set_ylabel('Monetary (£K)')

plt.tight_layout()
plt.savefig('multivariate_rfm.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 5D. Revenue Treemap: Country × Top Products ───────────────
treemap_data = df_clean.groupby(['Country','Description'])['Revenue'].sum().reset_index()
treemap_data = treemap_data[treemap_data['Country'].isin(
    df_clean.groupby('Country')['Revenue'].sum().nlargest(8).index
)]
top_prod_per_country = treemap_data.groupby('Country').apply(
    lambda x: x.nlargest(5,'Revenue')
).reset_index(drop=True)

fig_tm = px.treemap(
    top_prod_per_country,
    path=['Country','Description'],
    values='Revenue',
    title='Revenue Treemap: Top 8 Countries × Top 5 Products',
    color='Revenue',
    color_continuous_scale='Blues',
)
fig_tm.update_layout(height=600)
fig_tm.show()

# ── 5E. Monthly cohort – first purchase month ─────────────────
print("\n📊 Monthly Revenue Trend with YoY comparison (Plotly):")
monthly_yr = df_clean.groupby(['Year','Month'])['Revenue'].sum().reset_index()
monthly_yr['Month_Name'] = monthly_yr['Month'].map(
    {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
     7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'})

fig_yr = px.line(
    monthly_yr, x='Month_Name', y='Revenue', color='Year',
    title='Monthly Revenue: Year-over-Year Comparison',
    markers=True,
    color_discrete_sequence=['#1A73E8','#EA4335','#34A853'],
    category_orders={'Month_Name':['Jan','Feb','Mar','Apr','May','Jun',
                                   'Jul','Aug','Sep','Oct','Nov','Dec']}
)
fig_yr.update_layout(height=400, yaxis_title='Revenue (£)',
                     legend=dict(x=0.01, y=0.99))
fig_yr.show()

# ── 5F. Top customer analysis ─────────────────────────────────
top_customers = df_clean[df_clean['CustomerID'] != 0].groupby('CustomerID').agg(
    Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'), Items=('Quantity','sum')
).sort_values('Revenue', ascending=False).head(20)

fig_cust, ax_cust = plt.subplots(figsize=(14, 5))
bars = ax_cust.bar(top_customers.index.astype(str), top_customers['Revenue']/1e3,
                   color='#1A73E8', edgecolor='white')
ax_cust.set_title('Top 20 Customers by Total Revenue', fontsize=14, fontweight='bold')
ax_cust.set_xlabel('Customer ID'); ax_cust.set_ylabel('Revenue (£K)')
ax_cust.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('multivariate_top_customers.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n✅ Cell 5 complete.")

---
## 💡 CELL 6: Business Insights & Recommendations

**Mục đích:** Tổng hợp key findings từ EDA, đánh giá impact của outlier treatment, và đưa ra recommendations cùng next steps.

In [ ]:
# ============================================================
# CELL 6: BUSINESS INSIGHTS & RECOMMENDATIONS
# ============================================================

print("=" * 65)
print("💡  BUSINESS INSIGHTS & RECOMMENDATIONS")
print("=" * 65)

# ── 6A. Automated insight extraction ─────────────────────────

# Revenue metrics
total_rev       = df_clean['Revenue'].sum()
avg_order_val   = df_clean.groupby('InvoiceNo')['Revenue'].sum().mean()
top_country     = df_clean.groupby('Country')['Revenue'].sum().idxmax()
top_country_pct = df_clean.groupby('Country')['Revenue'].sum().max() / total_rev * 100
top_product     = df_clean.groupby('Description')['Revenue'].sum().idxmax()

# Time insights
best_month_grp  = df_clean.groupby('Month')['Revenue'].sum()
best_month      = best_month_grp.idxmax()
best_month_name = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
                   7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
best_weekday    = df_clean.groupby('Weekday')['Revenue'].sum().idxmax()
best_hour       = df_clean.groupby('Hour')['Revenue'].sum().idxmax()

# Customer metrics
n_identified    = (df_clean['CustomerID'] != 0).sum()
n_guest         = (df_clean['CustomerID'] == 0).sum()
guest_pct       = n_guest / len(df_clean) * 100

# RFM metrics
champions_pct   = (rfm['Segment'] == '🌟 Champions').sum() / len(rfm) * 100
lost_pct        = (rfm['Segment'] == '🔴 Lost').sum()       / len(rfm) * 100

print("""
╔══════════════════════════════════════════════════════════════╗
║              📈  KEY BUSINESS FINDINGS                       ║
╚══════════════════════════════════════════════════════════════╝
""")

print("📦  REVENUE OVERVIEW")
print(f"   • Total Revenue              : £{total_rev:,.2f}")
print(f"   • Avg Order Value            : £{avg_order_val:,.2f}")
print(f"   • Top Country                : {top_country} ({top_country_pct:.1f}% of revenue)")
print(f"   • Top Product by Revenue     : {top_product[:60]}")

print("\n⏰  TEMPORAL PATTERNS")
print(f"   • Best Month                 : {best_month_name[best_month]} (highest revenue)")
print(f"   • Best Weekday               : {best_weekday}")
print(f"   • Peak Hour                  : {best_hour}:00 – {best_hour+1}:00")
print(f"   • Q4 Dominance               : Nov–Dec are likely strongest months (gift season)")

print("\n👥  CUSTOMER INSIGHTS")
print(f"   • Identified Customers       : {df_clean['CustomerID'].nunique():,} unique")
print(f"   • Guest Transactions         : {n_guest:,} ({guest_pct:.1f}% of total)")
print(f"   • Champion Customers (RFM)   : {champions_pct:.1f}% of customer base")
print(f"   • Lost Customers (RFM)       : {lost_pct:.1f}% of customer base")

print("\n🌍  GEOGRAPHIC")
top5_countries = df_clean.groupby('Country')['Revenue'].sum().nlargest(5)
for c, r in top5_countries.items():
    pct = r / total_rev * 100
    print(f"   • {c:25s}: £{r:>12,.0f} ({pct:.1f}%)")

print("""
╔══════════════════════════════════════════════════════════════╗
║          🧹  IMPACT OF OUTLIER TREATMENT                     ║
╚══════════════════════════════════════════════════════════════╝
""")
print("   ✅ Removing cancelled invoices → cleaner net sales picture")
print("   ✅ Removing negative quantity  → eliminates return noise from sales metrics")
print("   ✅ Removing zero-price rows    → revenue metrics no longer deflated")
print("   ✅ Capping extreme Quantity    → avg order stats are meaningful, not skewed")
print("   ✅ Capping extreme UnitPrice   → correlation patterns are accurate")

print("""
╔══════════════════════════════════════════════════════════════╗
║           🎯  STRATEGIC RECOMMENDATIONS                      ║
╚══════════════════════════════════════════════════════════════╝
""")
recommendations = [
    ("Focus on UK market retention",
     f"UK contributes ~{top_country_pct:.0f}% of revenue. Loyalty programs should prioritize UK customers."),
    ("Capitalize on seasonal peaks",
     f"Revenue spikes in {best_month_name[best_month]}. Plan inventory & campaigns 6–8 weeks ahead."),
    ("Re-engage At-Risk & Lost customers",
     f"{lost_pct:.1f}% of customers are lost. Targeted win-back campaigns (discount vouchers, emails)."),
    ("Promote peak-hour campaigns",
     f"Most transactions occur around {best_hour}:00. Schedule email/SMS marketing before peak hour."),
    ("International expansion",
     "Germany, France, Netherlands show strong signals. Consider localized campaigns."),
    ("Reduce guest checkouts",
     f"{guest_pct:.1f}% transactions are guest. Account creation incentives boost future CRM value."),
    ("VIP program for Champions",
     f"{champions_pct:.1f}% are Champions (high R+F+M). Create exclusive tiers to retain them."),
]
for i, (title, detail) in enumerate(recommendations, 1):
    print(f"   {i}. {title}")
    print(f"      → {detail}")
    print()

print("""
╔══════════════════════════════════════════════════════════════╗
║           🔭  NEXT STEPS FOR DEEPER ANALYSIS                 ║
╚══════════════════════════════════════════════════════════════╝
""")
next_steps = [
    "📌 Step 4 – Customer Segmentation: K-Means clustering on RFM scores",
    "📌 Step 5 – Time Series Forecasting: ARIMA / Prophet on monthly revenue",
    "📌 Step 6 – Market Basket Analysis: Apriori / FP-Growth for product associations",
    "📌 Step 7 – Churn Prediction: Classification model on RFM features",
    "📌 Step 8 – Geo Analysis: Folium/Plotly map of revenue by country",
    "📌 Step 9 – Cohort Analysis: Monthly retention heatmap by first-purchase cohort",
    "📌 Step 10 – Dashboard: Build interactive Plotly Dash or Streamlit app",
]
for step in next_steps:
    print(f"   {step}")

# ── 6B. Final KPI summary card ────────────────────────────────
print("\n" + "=" * 65)
print("📋 FINAL EDA KPI SUMMARY")
print("=" * 65)
kpi = {
    'Total Revenue (£)'         : f"{total_rev:,.2f}",
    'Total Invoices'            : f"{df_clean['InvoiceNo'].nunique():,}",
    'Total Customers'           : f"{df_clean['CustomerID'].nunique():,}",
    'Total Products'            : f"{df_clean['StockCode'].nunique():,}",
    'Countries'                 : f"{df_clean['Country'].nunique()}",
    'Avg Order Value (£)'       : f"{avg_order_val:,.2f}",
    'Best Month'                : best_month_name[best_month],
    'Best Weekday'              : best_weekday,
    'Peak Hour'                 : f"{best_hour}:00",
    'Top Country'               : top_country,
    'Champion Customers (%)'    : f"{champions_pct:.1f}%",
    'Guest Transactions (%)'    : f"{guest_pct:.1f}%",
}
kpi_df = pd.DataFrame(list(kpi.items()), columns=['KPI','Value'])
display(kpi_df.style.set_properties(**{'text-align':'left'}).hide(axis='index'))

print("\n🎉 Step 3 EDA Complete! df_clean is ready for modeling & dashboarding.")